# 📊 Analyse Exploratoire des Prix

Notebook pour explorer et analyser les données extraites

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuration
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Charger les données

In [ ]:
# Charger données extraites
data_path = Path('../data/processed/prices_extracted.csv')

if data_path.exists():
    df = pd.read_csv(data_path)
    df['date'] = pd.to_datetime(df['date'])
    print(f"✓ {len(df)} prix chargés")
    df.head()
else:
    print("❌ Fichier non trouvé. Lancez d'abord l'extraction.")
    df = pd.DataFrame()

## 2. Statistiques descriptives

In [ ]:
# Stats globales
df.describe()

In [ ]:
# Distribution par type d'animal
df['animal_type'].value_counts()

## 3. Visualisations

In [ ]:
# Distribution des prix
plt.figure(figsize=(12, 6))
sns.histplot(df['prix'], bins=30, kde=True)
plt.title('Distribution des Prix')
plt.xlabel('Prix (FCFA)')
plt.show()

In [ ]:
# Prix par type d'animal
animal_data = df[df['animal_type'] != 'non_specifie']

if len(animal_data) > 0:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=animal_data, x='animal_type', y='prix')
    plt.title('Prix par Type d\'Animal')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Évolution temporelle
if len(df) > 10:
    monthly = df.set_index('date').resample('M')['prix'].mean()
    
    plt.figure(figsize=(12, 6))
    monthly.plot(marker='o')
    plt.title('Évolution Mensuelle des Prix')
    plt.xlabel('Date')
    plt.ylabel('Prix Moyen (FCFA)')
    plt.grid(True)
    plt.show()

## 4. Analyse par type d'animal

In [ ]:
# Stats par animal
animal_stats = df.groupby('animal_type')['prix'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
animal_stats.round(0)

## 5. Détection d'opportunités simples

In [ ]:
# Prix bas (< Q1) par type
opportunities = []

for animal in df['animal_type'].unique():
    if animal == 'non_specifie':
        continue
    
    animal_df = df[df['animal_type'] == animal]
    if len(animal_df) < 5:
        continue
    
    Q1 = animal_df['prix'].quantile(0.25)
    median = animal_df['prix'].median()
    
    good_deals = animal_df[animal_df['prix'] < Q1]
    
    for _, deal in good_deals.iterrows():
        reduction = (median - deal['prix']) / median * 100
        opportunities.append({
            'animal': animal,
            'prix': deal['prix'],
            'median': median,
            'reduction_pct': reduction
        })

if opportunities:
    opp_df = pd.DataFrame(opportunities)
    opp_df = opp_df.sort_values('reduction_pct', ascending=False)
    print(f"\n🎯 {len(opp_df)} opportunités détectées:\n")
    print(opp_df.head(10))
else:
    print("Pas assez de données pour détecter des opportunités")

## 6. Export résultats

In [ ]:
# Sauvegarder analyse
analysis_path = Path('../data/processed/analysis_results.csv')
animal_stats.to_csv(analysis_path)
print(f"✓ Analyse sauvegardée: {analysis_path}")

---

## À explorer ensuite:

- Corrélations entre variables
- Patterns saisonniers
- Analyse vendeurs récurrents
- Clustering des prix
- Tests statistiques